# Bangladesh Labour Act QA — full training run + ablation study

This notebook supersedes `finetune_llama32_3b.ipynb` for the paper's results.

The original run was a **pilot**: `max_steps=100` (about 466 s on a T4), no early
stopping, and it trained on the *entire* dataset, so no examples were reserved
for testing. This notebook fixes all three and adds the ablation the reviewer
asked for:

| | Pilot notebook | This notebook |
|---|---|---|
| Training data | whole dataset (leakage into test) | `train_final.json` only, held-out set removed |
| Length | `max_steps=100` | full epochs with early stopping on eval loss |
| Model selection | last step | best checkpoint (`load_best_model_at_end`) |
| Seeds / versions | not recorded | captured to `run_manifest.json` |
| Ablation | none | dataset size × LoRA rank × refusal data |

**Runtime:** the main run is roughly 45–90 min on a T4; the full ablation grid
adds about 2–4 h. You can run Section 7 (main run) alone and come back to the
ablation later.

**What to send back:** the `runs/` folder — it contains one `trainer_state.json`
per configuration, plus `run_manifest.json` and `ablation_results.json`. Those
files are what the paper's training and ablation tables are generated from.

## 1. Install dependencies

Versions are pinned so the run is reproducible and the paper can state them.

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try:
        import numpy, PIL
        get_numpy = f"numpy=={numpy.__version__}"; get_pil = f"pillow=={PIL.__version__}"
    except Exception:
        get_numpy = "numpy"; get_pil = "pillow"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {get_numpy} {get_pil} torchvision bitsandbytes \
        "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2
!pip install -qqq datasets huggingface_hub

## 2. Seeds, environment, and hardware capture

Everything the paper's reproducibility paragraph needs is recorded here.

In [ ]:
import json, os, platform, random, subprocess, sys
from pathlib import Path

SEED = 3407

import numpy as np
import torch

def set_all_seeds(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_all_seeds()

RUNS = Path("runs"); RUNS.mkdir(exist_ok=True)

def pkg_version(name):
    try:
        return __import__("importlib.metadata", fromlist=["version"]).version(name)
    except Exception:
        return None

gpu = torch.cuda.get_device_properties(0) if torch.cuda.is_available() else None
MANIFEST = {
    "seed": SEED,
    "python": platform.python_version(),
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu_name": gpu.name if gpu else None,
    "gpu_total_gb": round(gpu.total_memory / 1024**3, 2) if gpu else None,
    "packages": {p: pkg_version(p) for p in
                 ["transformers", "trl", "peft", "unsloth", "datasets",
                  "bitsandbytes", "accelerate", "torch"]},
}
json.dump(MANIFEST, open(RUNS / "run_manifest.json", "w"), indent=1)
print(json.dumps(MANIFEST, indent=1))

## 3. Load the leakage-free training pool

Upload **`data/final/train_final.json`** (3 119 examples) and, for the leakage
assertion, **`data/eval/heldout_test.json`** and **`data/eval/scenario_test.json`**.

`train_final.json` was produced by `scripts/build_test_split.py`, which carved the
held-out set out of the cleaned dataset first. The assertion below re-verifies
that on this machine — a training run that silently included test questions would
invalidate every number in the paper, so it is checked rather than assumed.

In [ ]:
from google.colab import files

print("Upload train_final.json, heldout_test.json, scenario_test.json")
uploaded = files.upload()

def load(name):
    for k in uploaded:
        if name in k:
            return json.loads(uploaded[k].decode("utf-8"))
    raise SystemExit(f"{name} not uploaded")

raw_train = load("train_final")
heldout   = load("heldout_test")
scenario  = load("scenario_test")
print(f"train={len(raw_train)}  heldout={len(heldout)}  scenario={len(scenario)}")

In [ ]:
import hashlib, re

def norm_q(q):
    return re.sub(r"[^a-z0-9]+", " ", (q or "").lower()).strip()

def qhash(q):
    return hashlib.sha1(norm_q(q).encode()).hexdigest()

def train_question(sample):
    for m in sample["messages"]:
        if m["role"] == "user":
            return m["content"]
    return ""

train_hashes = {qhash(train_question(s)) for s in raw_train}
for name, items in [("heldout", heldout), ("scenario", scenario)]:
    overlap = train_hashes & {qhash(i["question"]) for i in items}
    print(f"{name}: {len(overlap)} overlapping questions")
    assert not overlap, f"LEAKAGE: {len(overlap)} {name} questions appear in training data"
print("\nNo leakage: training pool and test sets are disjoint.")

## 4. Build the datasets

A fixed-seed 95/5 split of the training pool gives the validation set that early stopping monitors. The held-out and scenario sets are never touched here — they are only used by `scripts/evaluate.py` after training.

In [ ]:
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

def build_dataset(samples, tokenizer):
    rows = []
    for s in samples:
        msgs = s.get("messages") or s.get("conversations")
        if not msgs:
            continue
        rows.append({"text": tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=False)})
    return Dataset.from_list(rows)

def split_train_val(samples, val_frac=0.05, seed=SEED):
    idx = list(range(len(samples)))
    random.Random(seed).shuffle(idx)
    n_val = max(1, int(len(idx) * val_frac))
    val = [samples[i] for i in idx[:n_val]]
    tr  = [samples[i] for i in idx[n_val:]]
    return tr, val

train_samples, val_samples = split_train_val(raw_train)
print(f"train={len(train_samples)}  val={len(val_samples)}")

## 5. Refusal examples (for the ablation)

`refusal_examples.json` teaches the model to decline questions outside the Act.
It is part of the main training mix; the ablation measures what removing it costs
on the out-of-scope test set. Upload it if you want to run that arm.

In [ ]:
try:
    up2 = files.upload()
    refusal = json.loads(list(up2.values())[0].decode("utf-8"))
    print(f"refusal examples: {len(refusal)}")
except Exception as e:
    refusal = []
    print("no refusal file loaded ->", e)

# train_final.json already contains the refusal examples; this identifies them so
# the 'no refusal data' ablation arm can strip them back out.
refusal_hashes = {qhash(train_question(s)) for s in refusal}
print(f"{sum(1 for s in train_samples if qhash(train_question(s)) in refusal_hashes)}"
      " refusal examples present in the training split")

## 6. The experiment runner

One function trains one configuration and writes its `trainer_state.json`. Every
arm of the ablation goes through it, so the arms differ only in the variables
being ablated.

Early stopping watches validation loss with patience 3 and restores the best
checkpoint, so results do not depend on where an arbitrary step limit landed.

In [ ]:
import gc, time
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

MAX_SEQ_LEN = 2048

def run_experiment(name, samples, lora_r=16, epochs=3, lr=2e-4,
                   val=None, max_steps=-1):
    """Train one configuration; return its metrics and save trainer_state.json."""
    print(f"\n{'='*60}\nRUN: {name}  (n={len(samples)}, r={lora_r}, epochs={epochs})\n{'='*60}")
    set_all_seeds()
    t0 = time.time()

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/Llama-3.2-3B-Instruct",
        max_seq_length=MAX_SEQ_LEN, dtype=None, load_in_4bit=True,
    )
    tokenizer = get_chat_template(tokenizer, chat_template="llama-3.2")
    model = FastLanguageModel.get_peft_model(
        model, r=lora_r,
        target_modules=["q_proj","k_proj","v_proj","o_proj",
                        "gate_proj","up_proj","down_proj"],
        lora_alpha=lora_r, lora_dropout=0, bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED, use_rslora=False, loftq_config=None,
    )

    out_dir = str(RUNS / name)
    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer,
        train_dataset=build_dataset(samples, tokenizer),
        eval_dataset=build_dataset(val if val is not None else val_samples, tokenizer),
        args=SFTConfig(
            output_dir=out_dir,
            dataset_text_field="text", max_seq_length=MAX_SEQ_LEN,
            dataset_num_proc=2, packing=False,
            per_device_train_batch_size=2, per_device_eval_batch_size=2,
            gradient_accumulation_steps=4, warmup_ratio=0.03,
            num_train_epochs=epochs, max_steps=max_steps,
            learning_rate=lr,
            fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
            logging_steps=10,
            eval_strategy="steps", eval_steps=50,
            save_strategy="steps", save_steps=50, save_total_limit=1,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss", greater_is_better=False,
            optim="adamw_8bit", weight_decay=0.01,
            lr_scheduler_type="linear", seed=SEED, data_seed=SEED,
            report_to="none",
        ),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    )

    stats = trainer.train()
    metrics = trainer.evaluate()
    peak_gb = round(torch.cuda.max_memory_reserved() / 1024**3, 3)

    result = {
        "name": name, "n_train": len(samples), "lora_r": lora_r,
        "epochs": epochs, "lr": lr,
        "train_loss": stats.metrics.get("train_loss"),
        "train_runtime_s": stats.metrics.get("train_runtime"),
        "eval_loss": metrics.get("eval_loss"),
        "perplexity": float(np.exp(metrics["eval_loss"])) if "eval_loss" in metrics else None,
        "global_step": int(trainer.state.global_step),
        "best_checkpoint_step": trainer.state.best_model_checkpoint,
        "best_eval_loss": trainer.state.best_metric,
        "peak_gpu_gb": peak_gb,
        "wall_clock_s": round(time.time() - t0, 1),
    }

    Path(out_dir).mkdir(parents=True, exist_ok=True)
    trainer.state.save_to_json(f"{out_dir}/trainer_state.json")
    json.dump(result, open(f"{out_dir}/result.json", "w"), indent=1)
    print(json.dumps(result, indent=1))

    del trainer, model
    gc.collect(); torch.cuda.empty_cache()
    return result

## 7. Main run

The configuration reported as the paper's primary model: full training pool,
LoRA rank 16, up to 3 epochs with early stopping.

In [ ]:
main_result = run_experiment("main_full", train_samples, lora_r=16, epochs=3)

## 8. Ablation study

Three questions, one variable at a time:

1. **How much data is needed?** 25 % / 50 % / 100 % of the training pool.
2. **Does LoRA capacity matter?** rank 8 / 16 / 32.
3. **Do the refusal examples earn their place?** with vs without them.

Arms 1 and 2 are cheap to interpret because everything else is held fixed. Arm 3
is the one that matters for safety: its effect shows up on the out-of-scope test
set, which you measure afterwards with `scripts/evaluate.py`, not here.

In [ ]:
ablation = []

# --- 1. dataset size -------------------------------------------------------
rng = random.Random(SEED)
for frac in (0.25, 0.50):
    n = int(len(train_samples) * frac)
    subset = rng.sample(train_samples, n)
    ablation.append(run_experiment(f"data_{int(frac*100)}pct", subset,
                                   lora_r=16, epochs=3))

# 100% arm is the main run; reuse it rather than retraining.
ablation.append({**main_result, "name": "data_100pct"})

In [ ]:
# --- 2. LoRA rank ----------------------------------------------------------
for r in (8, 32):
    ablation.append(run_experiment(f"lora_r{r}", train_samples, lora_r=r, epochs=3))
ablation.append({**main_result, "name": "lora_r16"})

In [ ]:
# --- 3. refusal examples ---------------------------------------------------
if refusal_hashes:
    no_refusal = [s for s in train_samples
                  if qhash(train_question(s)) not in refusal_hashes]
    print(f"removed {len(train_samples) - len(no_refusal)} refusal examples")
    ablation.append(run_experiment("no_refusal_data", no_refusal,
                                   lora_r=16, epochs=3))
    ablation.append({**main_result, "name": "with_refusal_data"})
else:
    print("skipped: refusal_examples.json was not uploaded")

In [ ]:
json.dump(ablation, open(RUNS / "ablation_results.json", "w"), indent=1)

print(f"{'run':<22}{'n_train':>8}{'r':>4}{'steps':>7}{'eval_loss':>11}{'ppl':>8}")
for r in ablation:
    print(f"{r['name']:<22}{r['n_train']:>8}{r['lora_r']:>4}{r['global_step']:>7}"
          f"{(r['eval_loss'] or 0):>11.4f}{(r['perplexity'] or 0):>8.2f}")

## 9. Export the main model

Only the main run is exported for deployment. Re-run Section 7 first if you ran
ablation arms afterwards, so the exported adapter is the main configuration.

In [ ]:
EXPORT = False   # set True to re-export the deployable model

if EXPORT:
    set_all_seeds()
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/Llama-3.2-3B-Instruct",
        max_seq_length=MAX_SEQ_LEN, dtype=None, load_in_4bit=True)
    tokenizer = get_chat_template(tokenizer, chat_template="llama-3.2")
    model = FastLanguageModel.get_peft_model(
        model, r=16,
        target_modules=["q_proj","k_proj","v_proj","o_proj",
                        "gate_proj","up_proj","down_proj"],
        lora_alpha=16, lora_dropout=0, bias="none",
        use_gradient_checkpointing="unsloth", random_state=SEED)
    model.load_adapter(str(RUNS / "main_full"), adapter_name="default")
    model.save_pretrained("hr-persona-bd-llama32-3b-lora")
    tokenizer.save_pretrained("hr-persona-bd-llama32-3b-lora")
    model.save_pretrained_gguf("hr-persona-bd-llama32-3b-gguf", tokenizer,
                               quantization_method="q4_k_m")
    print("exported")

## 10. Send the results back

Download `runs.zip` and hand it back. It contains `run_manifest.json` (versions,
seeds, hardware), one `trainer_state.json` + `result.json` per configuration, and
`ablation_results.json`.

`scripts/publication_content.py` reads these to generate the paper's training and
ablation tables, so no number is transcribed by hand.

In [ ]:
!zip -qr runs.zip runs/
files.download("runs.zip")